# Create Work Links (oxjob #1490)

One row per work-to-work link: `origin_work_id`, `target_work_id`, `relationship` (a `/relationships` value). A plain
union of tables that are already curated, with no rule of its own (Casey, 20 Nov 2026): the served rows of
`work_citations` (#2314) as `citation`, each work's `parent_works` from `work_parents` (#2065) with its part-of kind,
and the served rows of `work_review_links` (#2481, built by this job's first task, `CreateWorkReviewLinks`) as `review`, and the served rows of `work_pull_notice_links` (#2264, built by `CreateWorkPullNoticeLinks`)
under their link type: `retraction`, `withdrawal` or `removal`, from the notice to the work it pulls.
Built after `Publish_Works`, so links change only on a night that published. `notebooks/elastic/sync_links` sends the
changes to the `links` search index. Plan: oxjobs `working/references-entity/LINKS_TABLE_PLAN.md`.

In [ ]:
CREATE OR REPLACE TABLE openalex.works.work_links AS
SELECT citing_work_id AS origin_work_id, cited_work_id AS target_work_id, 'citation' AS relationship
FROM openalex.works.work_citations
WHERE served
UNION ALL
SELECT p.work_id AS origin_work_id,
       CAST(regexp_extract(pw.id, 'W([0-9]+)$', 1) AS BIGINT) AS target_work_id,
       pw.relationship
FROM openalex.works.work_parents p
LATERAL VIEW explode(p.parent_works) AS pw
UNION ALL
SELECT origin_work_id, target_work_id, 'review' AS relationship
FROM openalex.works.work_review_links
WHERE served
UNION ALL
-- oxjob #2264: pull notices, from the notice (origin) to the work it retracts, withdraws or removes (target).
SELECT origin_work_id, target_work_id, link_type AS relationship
FROM openalex.works.work_pull_notice_links
WHERE served